**1. Setup**

In [1]:
%pip install -q numpy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns",None)

In [3]:
train_df=pd.read_csv("titanic_train_raw.csv")
test_df=pd.read_csv("titanic_test_raw.csv")

In [4]:
train_df.shape

(712, 14)

In [6]:
test_df.shape

(179, 14)

3 Separte x and y

In [7]:
target="survived"

In [8]:
#training set -feature and target column
X_train=train_df.drop(columns=[target])
y_train=train_df[target]

#test set -feature and target column
X_test=test_df.drop(columns=[target])
y_test=test_df[target]



In [9]:
X_train.head()

,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone
0,3,male,NaN,0,0,56.4958,S,Third,man,True,NaN,Southampton,True
1,2,male,NaN,0,0,0.0000,S,Second,man,True,NaN,Southampton,True
2,1,male,NaN,0,0,221.7792,S,First,man,True,C,Southampton,True
3,3,female,18.0,0,1,9.3500,S,Third,woman,False,NaN,Southampton,False
4,2,female,31.0,1,1,26.2500,S,Second,woman,False,NaN,Southampton,False


In [10]:
y_test.head()

0    0
1    0
2    1
3    0
4    1
Name: survived, dtype: int64

In [15]:
print("Missing values and train")
print(X_train.isna().sum())
print("-"*50)
print(X_test.isna().sum())


Missing values and train
pclass           0
sex              0
age            137
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           553
embark_town      2
alone            0
dtype: int64
--------------------------------------------------
pclass           0
sex              0
age             40
sibsp            0
parch            0
fare             0
embarked         0
class            0
who              0
adult_male       0
deck           135
embark_town      0
alone            0
dtype: int64


In [16]:
print("Missing values and train")
print(X_train.isna().mean()*100)
print("-"*50)
print(X_test.isna().mean()*100)


Missing values and train
pclass          0.000000
sex             0.000000
age            19.241573
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.280899
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.668539
embark_town     0.280899
alone           0.000000
dtype: float64
--------------------------------------------------
pclass          0.000000
sex             0.000000
age            22.346369
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.000000
class           0.000000
who             0.000000
adult_male      0.000000
deck           75.418994
embark_town     0.000000
alone           0.000000
dtype: float64


If having more missing value in a column then dropping a row is bad idea

In [17]:
print("Demonstrate of dropna:")
print(X_train.shape)


#make sure to drop the corresponding row in x and y
demo_df=X_train.dropna()
print(demo_df.shape)

Demonstrate of dropna:
(712, 13)
(141, 13)


dropna removed all rows that had even one missing vlaue

In [19]:
missing_threshold=40
missing_percent_train=X_train.isna().mean()*100
print(missing_percent_train)

high_missing_cols=missing_percent_train[missing_percent_train>missing_threshold].index.tolist()
print("Columns with more then 40% missing values",high_missing_cols)

pclass          0.000000
sex             0.000000
age            19.241573
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.280899
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.668539
embark_town     0.280899
alone           0.000000
dtype: float64
Columns with more then 40% missing values ['deck']


In [ ]:
# drop the entire column here droppping the deck columns


# X_train=X_train.drop(columns=high_missing_cols)
# X_test=X_test.drop(columns=high_missing_cols)


7 . Identify numerical and Categorical columns

In [20]:
cols_list=X_train.columns.to_list()
print(cols_list)

['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alone']


In [21]:
nums_cols=["age","sibsp","parch","fare"]
cat_cols=['pclass', 'sex','embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alone']
print(nums_cols)
print(cat_cols)

['age', 'sibsp', 'parch', 'fare']
['pclass', 'sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alone']


**Impute missing values the correct way (Train only)**

In [27]:
X_train_imputed=X_train.copy(deep=True)
X_test_imputed=X_test.copy()

In [23]:
X_train_imputed.head()

,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone
0,3,male,NaN,0,0,56.4958,S,Third,man,True,NaN,Southampton,True
1,2,male,NaN,0,0,0.0000,S,Second,man,True,NaN,Southampton,True
2,1,male,NaN,0,0,221.7792,S,First,man,True,C,Southampton,True
3,3,female,18.0,0,1,9.3500,S,Third,woman,False,NaN,Southampton,False
4,2,female,31.0,1,1,26.2500,S,Second,woman,False,NaN,Southampton,False


Numeric Imputation

In [30]:
X_train[nums_cols].isna().sum()

age      137
sibsp      0
parch      0
fare       0
dtype: int64

In [34]:
numeric_medians={}
for col in nums_cols:
    # mean_val=X_train_imputed[col].mean()
    median_val=X_train_imputed[col].median()
    numeric_medians[col]=median_val
    print(f"filling numeric columnss: {col} with train median: {median_val}")
    
    X_train_imputed[col]=X_train_imputed[col].fillna(median_val)
    X_test_imputed[col]=X_test_imputed[col].fillna(median_val)
    
    
numeric_medians
    
    
    

filling numeric columnss: age with train median: 28.5
filling numeric columnss: sibsp with train median: 0.0
filling numeric columnss: parch with train median: 0.0
filling numeric columnss: fare with train median: 14.4542


{'age': np.float64(28.5),
 'sibsp': np.float64(0.0),
 'parch': np.float64(0.0),
 'fare': np.float64(14.4542)}

 Categorical value imputaton
 

In [39]:
categorical_modes={}
for col in cat_cols:
    mode_val=X_train_imputed[col].mode().iloc[0]
    categorical_modes[col]=mode_val
    print(f"filling categorical columns:{col}, with mode {mode_val}")
    
    X_train_imputed[col]=X_train_imputed[col].fillna(mode_val)
    X_test_imputed[col]=X_test_imputed[col].fillna(mode_val)
    
    categorical_modes
    

filling categorical columns:pclass, with mode 3
filling categorical columns:sex, with mode male
filling categorical columns:embarked, with mode S
filling categorical columns:class, with mode Third
filling categorical columns:who, with mode man
filling categorical columns:adult_male, with mode True
filling categorical columns:deck, with mode C
filling categorical columns:embark_town, with mode Southampton
filling categorical columns:alone, with mode True


Recombine featues and target for future

In [43]:
train_imputed_df=X_train_imputed.copy()
test_imputed_df=X_test_imputed.copy()

train_imputed_df[target]= y_train.values
test_imputed_df[target]= y_test.values


In [44]:
train_imputed_df.to_csv("titanic_train_imputed.csv")
test_imputed_df.to_csv("titanic_train_imputed.csv")
